# Mordred Descriptor Selection with mRMR

This notebook generates and preprocesses Mordred molecular descriptors, removes descriptors with excessive missing values or high correlation, and applies minimum Redundancy Maximum Relevance (mRMR) feature selection. XGBoost models are then used to compare different numbers of selected descriptors, tune the classification threshold on the validation set, and choose the final Mordred feature set before evaluation on the held-out scaffold test set.

In [ ]:
%pip install -q mordredcommunity mrmr-selection

In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem
from mordred import Calculator, descriptors
from mrmr import mrmr_classif

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Mordred and mRMR imported successfully.")

In [ ]:
data_df = pd.read_csv("preprocessed_model_data.csv")

print("Dataset shape:", data_df.shape)

print("\nAvailable columns:")
print(data_df.columns.tolist())

print("\nFirst five rows:")
display(data_df.head())

In [ ]:
# Convert SMILES into molecule objects required by Mordred

data_df["molecule"] = data_df[
    "canonical_smiles"
].apply(Chem.MolFromSmiles)

invalid_molecules = data_df["molecule"].isna().sum()

print("Total molecules:", len(data_df))
print("Valid molecules:", len(data_df) - invalid_molecules)
print("Invalid molecules:", invalid_molecules)

print("\nSplit counts:")
print(data_df["split"].value_counts())

In [ ]:
# Initialise Mordred using only 2D descriptors

mordred_calculator = Calculator(
    descriptors,
    ignore_3D=True
)

mordred_descriptor_names = [
    str(descriptor)
    for descriptor in mordred_calculator.descriptors
]

print(
    "Number of Mordred descriptors:",
    len(mordred_descriptor_names)
)

print("\nFirst 20 descriptor names:")
print(mordred_descriptor_names[:20])

In [ ]:
import time

sample_molecules = data_df["molecule"].iloc[:100].tolist()

start_time = time.time()

mordred_sample_df = mordred_calculator.pandas(
    sample_molecules,
    nproc=1,
    quiet=True
)

elapsed_time = time.time() - start_time

print("Sample descriptor shape:", mordred_sample_df.shape)
print(f"Time taken for 100 molecules: {elapsed_time:.2f} seconds")

display(mordred_sample_df.iloc[:5, :15])

In [ ]:
import os
import psutil

cpu_cores = os.cpu_count()
available_memory_gb = psutil.virtual_memory().available / (1024 ** 3)
total_memory_gb = psutil.virtual_memory().total / (1024 ** 3)

print("Available CPU cores:", cpu_cores)
print(f"Available memory: {available_memory_gb:.2f} GB")
print(f"Total memory: {total_memory_gb:.2f} GB")

In [ ]:
from pathlib import Path
import time
import gc

# Folder in which each completed chunk will be saved
chunk_folder = Path("mordred_chunks")
chunk_folder.mkdir(exist_ok=True)

# Small chunks prevent the 15 GB memory limit from being exceeded
chunk_size = 1000
number_of_processes = 4

total_molecules = len(data_df)

start_time = time.time()

for start_index in range(0, total_molecules, chunk_size):

    end_index = min(
        start_index + chunk_size,
        total_molecules
    )

    output_file = chunk_folder / (
        f"mordred_{start_index:06d}_{end_index:06d}.pkl"
    )

    # Allows the calculation to resume after a crash
    if output_file.exists():
        print(
            f"Already completed: "
            f"{start_index:,}–{end_index:,}"
        )
        continue

    print(
        f"Calculating molecules "
        f"{start_index:,}–{end_index:,}..."
    )

    molecule_chunk = (
        data_df["molecule"]
        .iloc[start_index:end_index]
        .tolist()
    )

    mordred_chunk_df = mordred_calculator.pandas(
        molecule_chunk,
        nproc=number_of_processes,
        quiet=True
    )

    # Preserve the original molecule row positions
    mordred_chunk_df.index = range(
        start_index,
        end_index
    )

    mordred_chunk_df.to_pickle(
        output_file
    )

    print(
        f"Saved: {output_file.name}"
    )

    # Release memory before starting the next chunk
    del molecule_chunk
    del mordred_chunk_df
    gc.collect()


elapsed_minutes = (
    time.time() - start_time
) / 60

print(
    f"\nChunk calculation finished in "
    f"{elapsed_minutes:.2f} minutes."
)

print(
    "Number of saved chunks:",
    len(list(chunk_folder.glob("mordred_*.pkl")))
)

In [ ]:
from pathlib import Path
import pandas as pd
import gc

chunk_files = sorted(
    Path("mordred_chunks").glob("mordred_*.pkl")
)

chunk_summary = []
total_rows = 0

expected_columns = None
columns_consistent = True

for file_path in chunk_files:

    chunk_df = pd.read_pickle(file_path)

    if expected_columns is None:
        expected_columns = chunk_df.columns.tolist()

    elif chunk_df.columns.tolist() != expected_columns:
        columns_consistent = False

    chunk_summary.append({
        "file": file_path.name,
        "rows": len(chunk_df),
        "columns": chunk_df.shape[1],
        "first_index": chunk_df.index.min(),
        "last_index": chunk_df.index.max()
    })

    total_rows += len(chunk_df)

    del chunk_df
    gc.collect()


chunk_summary_df = pd.DataFrame(chunk_summary)

print("Number of chunk files:", len(chunk_files))
print("Total rows across chunks:", total_rows)
print("Expected total rows:", len(data_df))
print("Number of Mordred descriptors:", len(expected_columns))
print("Columns consistent across chunks:", columns_consistent)

print("\nFirst three chunks:")
display(chunk_summary_df.head(3))

print("\nLast three chunks:")
display(chunk_summary_df.tail(3))

print(
    "\nAll molecules accounted for:",
    total_rows == len(data_df)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import gc

raw_chunk_files = sorted(
    Path("mordred_chunks").glob("mordred_*.pkl")
)

numeric_chunk_folder = Path(
    "mordred_numeric_chunks"
)

numeric_chunk_folder.mkdir(
    exist_ok=True
)

descriptor_columns = expected_columns

training_missing_counts = pd.Series(
    0,
    index=descriptor_columns,
    dtype="int64"
)

training_row_count = 0


for chunk_number, raw_file in enumerate(
    raw_chunk_files,
    start=1
):

    numeric_file = (
        numeric_chunk_folder /
        raw_file.name.replace(
            "mordred_",
            "mordred_numeric_"
        )
    )

    # Resume safely if a numeric chunk already exists
    if numeric_file.exists():

        numeric_chunk_df = pd.read_pickle(
            numeric_file
        )

    else:

        raw_chunk_df = pd.read_pickle(
            raw_file
        )

        # Mordred errors and missing objects become NaN
        numeric_chunk_df = raw_chunk_df.apply(
            pd.to_numeric,
            errors="coerce"
        )

        numeric_chunk_df = numeric_chunk_df.replace(
            [np.inf, -np.inf],
            np.nan
        )

        # Reduce memory usage
        numeric_chunk_df = numeric_chunk_df.astype(
            "float32"
        )

        numeric_chunk_df.to_pickle(
            numeric_file
        )

        del raw_chunk_df

    # Use only training rows to assess descriptor quality
    training_mask = (
        data_df.loc[
            numeric_chunk_df.index,
            "split"
        ]
        .eq("train")
        .to_numpy()
    )

    training_chunk_df = numeric_chunk_df.loc[
        training_mask
    ]

    training_missing_counts = (
        training_missing_counts.add(
            training_chunk_df.isna().sum(),
            fill_value=0
        )
        .astype("int64")
    )

    training_row_count += len(
        training_chunk_df
    )

    if (
        chunk_number % 10 == 0
        or chunk_number == len(raw_chunk_files)
    ):
        print(
            f"Processed {chunk_number}/"
            f"{len(raw_chunk_files)} chunks"
        )

    del numeric_chunk_df
    del training_chunk_df
    gc.collect()


mordred_missingness_df = pd.DataFrame({
    "descriptor": descriptor_columns,
    "missing_count_train":
        training_missing_counts.values
})

mordred_missingness_df[
    "missing_percent_train"
] = (
    mordred_missingness_df[
        "missing_count_train"
    ]
    / training_row_count
    * 100
)


mordred_missingness_df = (
    mordred_missingness_df
    .sort_values(
        "missing_percent_train",
        ascending=False
    )
    .reset_index(drop=True)
)


mordred_missingness_df.to_csv(
    "mordred_training_missingness.csv",
    index=False
)


print("\nTraining rows checked:", training_row_count)

print(
    "Descriptors with no missing values:",
    (
        mordred_missingness_df[
            "missing_count_train"
        ] == 0
    ).sum()
)

print(
    "Descriptors with up to 5% missing:",
    (
        mordred_missingness_df[
            "missing_percent_train"
        ] <= 5
    ).sum()
)

print(
    "Descriptors with more than 5% missing:",
    (
        mordred_missingness_df[
            "missing_percent_train"
        ] > 5
    ).sum()
)

print(
    "Descriptors completely missing:",
    (
        mordred_missingness_df[
            "missing_percent_train"
        ] == 100
    ).sum()
)

print("\nMost-missing descriptors:")
display(
    mordred_missingness_df.head(20)
)

print(
    "\nSaved: mordred_training_missingness.csv"
)
print(
    "Saved numeric chunks in: mordred_numeric_chunks/"
)
print(
    "\nThe test set has not been used for feature filtering."
)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

# Keep descriptors with no more than 5% missing values in training data
eligible_mordred_descriptors = (
    mordred_missingness_df.loc[
        mordred_missingness_df["missing_percent_train"] <= 5,
        "descriptor"
    ]
    .tolist()
)

print(
    "Descriptors retained after missingness filtering:",
    len(eligible_mordred_descriptors)
)


numeric_chunk_files = sorted(
    Path("mordred_numeric_chunks").glob(
        "mordred_numeric_*.pkl"
    )
)

training_parts = []

for chunk_number, file_path in enumerate(
    numeric_chunk_files,
    start=1
):

    chunk_df = pd.read_pickle(file_path)

    training_mask = (
        data_df.loc[
            chunk_df.index,
            "split"
        ]
        .eq("train")
        .to_numpy()
    )

    training_part = chunk_df.loc[
        training_mask,
        eligible_mordred_descriptors
    ].copy()

    training_parts.append(training_part)

    del chunk_df
    gc.collect()

    if (
        chunk_number % 10 == 0
        or chunk_number == len(numeric_chunk_files)
    ):
        print(
            f"Loaded {chunk_number}/"
            f"{len(numeric_chunk_files)} chunks"
        )


# Combine training rows
X_train_mordred = (
    pd.concat(training_parts)
    .sort_index()
)

del training_parts
gc.collect()


# Obtain training labels using the same row indices
y_train_mordred = (
    data_df.loc[
        X_train_mordred.index,
        "dual_candidate"
    ]
    .astype(int)
)


# Calculate medians using training data only
mordred_training_medians = (
    X_train_mordred.median(axis=0)
)


# Fill missing values with training medians
X_train_mordred = X_train_mordred.fillna(
    mordred_training_medians
)


# Remove constant descriptors using training data only
constant_mordred_descriptors = (
    X_train_mordred.columns[
        X_train_mordred.nunique(
            dropna=False
        ) <= 1
    ]
    .tolist()
)

usable_mordred_descriptors = [
    descriptor
    for descriptor in X_train_mordred.columns
    if descriptor not in constant_mordred_descriptors
]

X_train_mordred = (
    X_train_mordred[
        usable_mordred_descriptors
    ]
    .astype("float32")
)


# Save the cleaned training matrix and supporting files
X_train_mordred.to_pickle(
    "mordred_training_features_clean.pkl"
)

y_train_mordred.to_pickle(
    "mordred_training_labels.pkl"
)

pd.DataFrame({
    "descriptor": usable_mordred_descriptors
}).to_csv(
    "mordred_usable_descriptor_names.csv",
    index=False
)

mordred_training_medians[
    usable_mordred_descriptors
].rename(
    "training_median"
).to_csv(
    "mordred_training_medians.csv"
)


print("\nTraining feature shape:", X_train_mordred.shape)
print("Training label shape:", y_train_mordred.shape)

print(
    "Constant descriptors removed:",
    len(constant_mordred_descriptors)
)

print(
    "Final usable Mordred descriptors:",
    len(usable_mordred_descriptors)
)

print(
    "Remaining missing values:",
    int(X_train_mordred.isna().sum().sum())
)

print("\nSaved cleaned Mordred training files.")
print("The validation and test sets have not been used for filtering.")

In [ ]:
# Remove highly correlated Mordred descriptors using training data only

CORRELATION_THRESHOLD = 0.95

mordred_correlation_matrix = (
    X_train_mordred
    .corr(method="pearson")
    .abs()
)

upper_triangle = mordred_correlation_matrix.where(
    np.triu(
        np.ones(
            mordred_correlation_matrix.shape
        ),
        k=1
    ).astype(bool)
)

highly_correlated_mordred_descriptors = [
    column
    for column in upper_triangle.columns
    if (
        upper_triangle[column]
        > CORRELATION_THRESHOLD
    ).any()
]

mordred_descriptors_after_correlation = [
    column
    for column in X_train_mordred.columns
    if column not in highly_correlated_mordred_descriptors
]

X_train_mordred_filtered = (
    X_train_mordred[
        mordred_descriptors_after_correlation
    ]
    .copy()
    .astype("float32")
)

# Save filtered training features and descriptor names

X_train_mordred_filtered.to_pickle(
    "mordred_training_features_filtered.pkl"
)

pd.DataFrame({
    "descriptor":
        mordred_descriptors_after_correlation
}).to_csv(
    "mordred_filtered_descriptor_names.csv",
    index=False
)

pd.DataFrame({
    "removed_descriptor":
        highly_correlated_mordred_descriptors
}).to_csv(
    "mordred_highly_correlated_removed.csv",
    index=False
)

print(
    "Descriptors before correlation filtering:",
    X_train_mordred.shape[1]
)

print(
    "Highly correlated descriptors removed:",
    len(highly_correlated_mordred_descriptors)
)

print(
    "Descriptors remaining:",
    X_train_mordred_filtered.shape[1]
)

print(
    "Filtered training shape:",
    X_train_mordred_filtered.shape
)

print("\nSaved Mordred correlation-filtering files.")
print("The validation and test sets have not been used.")

In [ ]:
from pathlib import Path
import pandas as pd
import gc

validation_parts = []

for chunk_number, file_path in enumerate(
    numeric_chunk_files,
    start=1
):

    chunk_df = pd.read_pickle(file_path)

    validation_mask = (
        data_df.loc[
            chunk_df.index,
            "split"
        ]
        .eq("validation")
        .to_numpy()
    )

    validation_part = chunk_df.loc[
        validation_mask,
        mordred_descriptors_after_correlation
    ].copy()

    validation_parts.append(validation_part)

    del chunk_df
    gc.collect()

    if (
        chunk_number % 10 == 0
        or chunk_number == len(numeric_chunk_files)
    ):
        print(
            f"Loaded {chunk_number}/"
            f"{len(numeric_chunk_files)} chunks"
        )


# Combine validation rows

X_validation_mordred_filtered = (
    pd.concat(validation_parts)
    .sort_index()
)

del validation_parts
gc.collect()


# Fill missing values using training medians only

X_validation_mordred_filtered = (
    X_validation_mordred_filtered
    .fillna(
        mordred_training_medians[
            mordred_descriptors_after_correlation
        ]
    )
    .astype("float32")
)


# Validation labels

y_validation_mordred = (
    data_df.loc[
        X_validation_mordred_filtered.index,
        "dual_candidate"
    ]
    .astype(int)
)


# Save

X_validation_mordred_filtered.to_pickle(
    "mordred_validation_features_filtered.pkl"
)

y_validation_mordred.to_pickle(
    "mordred_validation_labels.pkl"
)


print(
    "\nValidation feature shape:",
    X_validation_mordred_filtered.shape
)

print(
    "Validation label shape:",
    y_validation_mordred.shape
)

print(
    "Remaining missing values:",
    int(
        X_validation_mordred_filtered
        .isna()
        .sum()
        .sum()
    )
)

print("\nValidation class counts:")
print(
    y_validation_mordred.value_counts()
    .sort_index()
)

print("\nSaved Mordred validation files.")
print("The test set has not been loaded or used.")

In [ ]:
from xgboost import XGBClassifier

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import pandas as pd
import joblib


# Balanced sample weights from training labels only
mordred_sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_mordred
)


# Same XGBoost settings used for the RDKit baseline
mordred_baseline_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)


mordred_baseline_model.fit(
    X_train_mordred_filtered,
    y_train_mordred,
    sample_weight=mordred_sample_weights,
    eval_set=[
        (
            X_validation_mordred_filtered,
            y_validation_mordred
        )
    ],
    verbose=False
)


# Validation probabilities and predictions at threshold 0.50
MORDRED_BASELINE_THRESHOLD = 0.50

mordred_validation_probability = (
    mordred_baseline_model.predict_proba(
        X_validation_mordred_filtered
    )[:, 1]
)

mordred_validation_prediction = (
    mordred_validation_probability
    >= MORDRED_BASELINE_THRESHOLD
).astype(int)


tn, fp, fn, tp = confusion_matrix(
    y_validation_mordred,
    mordred_validation_prediction
).ravel()

specificity = tn / (tn + fp)


mordred_baseline_results_df = pd.DataFrame([{
    "model": "Mordred XGBoost baseline",
    "number_of_features":
        X_train_mordred_filtered.shape[1],
    "threshold": MORDRED_BASELINE_THRESHOLD,

    "balanced_accuracy": balanced_accuracy_score(
        y_validation_mordred,
        mordred_validation_prediction
    ),

    "mcc": matthews_corrcoef(
        y_validation_mordred,
        mordred_validation_prediction
    ),

    "roc_auc": roc_auc_score(
        y_validation_mordred,
        mordred_validation_probability
    ),

    "average_precision": average_precision_score(
        y_validation_mordred,
        mordred_validation_probability
    ),

    "precision": precision_score(
        y_validation_mordred,
        mordred_validation_prediction,
        zero_division=0
    ),

    "recall": recall_score(
        y_validation_mordred,
        mordred_validation_prediction,
        zero_division=0
    ),

    "specificity": specificity,

    "f1_score": f1_score(
        y_validation_mordred,
        mordred_validation_prediction,
        zero_division=0
    ),

    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp
}])


display(
    mordred_baseline_results_df.round(3)
)

print(
    "Best boosting iteration:",
    mordred_baseline_model.best_iteration
)


# Save baseline model and results
joblib.dump(
    mordred_baseline_model,
    "mordred_xgboost_baseline_model.joblib"
)

mordred_baseline_results_df.to_csv(
    "mordred_xgboost_baseline_validation_results.csv",
    index=False
)

print("\nSaved Mordred baseline model and results.")
print("The test set has not been used.")

In [ ]:
import time
from mrmr import mrmr_classif

MRMR_MAX_FEATURES = 100

start_time = time.time()

mrmr_ranked_features = mrmr_classif(
    X=X_train_mordred_filtered,
    y=y_train_mordred,
    K=MRMR_MAX_FEATURES,
    relevance="f",
    redundancy="c",
    n_jobs=4,
    show_progress=True
)

elapsed_minutes = (
    time.time() - start_time
) / 60


mrmr_ranking_df = pd.DataFrame({
    "rank": range(
        1,
        len(mrmr_ranked_features) + 1
    ),
    "descriptor": mrmr_ranked_features
})


mrmr_ranking_df.to_csv(
    "mordred_mrmr_top100_ranking.csv",
    index=False
)


print(
    "Number of selected descriptors:",
    len(mrmr_ranked_features)
)

print(
    f"Time taken: {elapsed_minutes:.2f} minutes"
)

print("\nTop 20 mRMR descriptors:")
display(
    mrmr_ranking_df.head(20)
)

print(
    "\nSaved: mordred_mrmr_top100_ranking.csv"
)

print(
    "Only the training set was used for mRMR."
)

In [ ]:
import time
import joblib
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    confusion_matrix,
    f1_score
)


# Numbers of top-ranked mRMR descriptors to compare
mrmr_feature_counts = [
    20,
    40,
    60,
    80,
    100
]


mrmr_validation_results = []
mrmr_trained_models = {}

start_time = time.time()


for feature_count in mrmr_feature_counts:

    print(
        f"Training XGBoost using top "
        f"{feature_count} mRMR descriptors..."
    )

    # Select the first N descriptors from the mRMR ranking
    selected_features = mrmr_ranked_features[
        :feature_count
    ]

    X_train_selected = (
        X_train_mordred_filtered[
            selected_features
        ]
    )

    X_validation_selected = (
        X_validation_mordred_filtered[
            selected_features
        ]
    )


    # Use the same XGBoost settings as the Mordred baseline
    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=3,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.10,
        reg_lambda=1.00,
        tree_method="hist",
        eval_metric="logloss",
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_train_selected,
        y_train_mordred,
        sample_weight=mordred_sample_weights,
        eval_set=[
            (
                X_validation_selected,
                y_validation_mordred
            )
        ],
        verbose=False
    )


    # Validation probabilities
    validation_probability = (
        model.predict_proba(
            X_validation_selected
        )[:, 1]
    )


    # Initial comparison at threshold 0.50
    threshold = 0.50

    validation_prediction = (
        validation_probability >= threshold
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_validation_mordred,
        validation_prediction
    ).ravel()


    specificity = tn / (tn + fp)


    mrmr_validation_results.append({
        "model":
            f"Mordred mRMR XGBoost top {feature_count}",

        "number_of_features":
            feature_count,

        "threshold":
            threshold,

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation_mordred,
                validation_prediction
            ),

        "mcc":
            matthews_corrcoef(
                y_validation_mordred,
                validation_prediction
            ),

        "roc_auc":
            roc_auc_score(
                y_validation_mordred,
                validation_probability
            ),

        "average_precision":
            average_precision_score(
                y_validation_mordred,
                validation_probability
            ),

        "precision":
            precision_score(
                y_validation_mordred,
                validation_prediction,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_validation_mordred,
                validation_prediction,
                zero_division=0
            ),

        "specificity":
            specificity,

        "f1_score":
            f1_score(
                y_validation_mordred,
                validation_prediction,
                zero_division=0
            ),

        "true_negative":
            tn,

        "false_positive":
            fp,

        "false_negative":
            fn,

        "true_positive":
            tp,

        "best_iteration":
            model.best_iteration
    })


    # Keep model available inside the notebook
    mrmr_trained_models[
        feature_count
    ] = model


    # Save each trained model
    joblib.dump(
        model,
        (
            f"mordred_mrmr_xgboost_"
            f"top{feature_count}_model.joblib"
        )
    )


    # Save the descriptor names used by each model
    pd.DataFrame({
        "rank": range(
            1,
            feature_count + 1
        ),
        "descriptor": selected_features
    }).to_csv(
        (
            f"mordred_mrmr_top"
            f"{feature_count}_descriptors.csv"
        ),
        index=False
    )


    print(
        f"Completed top {feature_count} features."
    )


# Create final comparison table
mrmr_validation_results_df = pd.DataFrame(
    mrmr_validation_results
)


mrmr_validation_results_df = (
    mrmr_validation_results_df
    .sort_values(
        by=[
            "balanced_accuracy",
            "mcc"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


# Save results
mrmr_validation_results_df.to_csv(
    "mordred_mrmr_xgboost_validation_results.csv",
    index=False
)


display(
    mrmr_validation_results_df.round(3)
)


elapsed_minutes = (
    time.time() - start_time
) / 60


print(
    f"\nTotal time: "
    f"{elapsed_minutes:.2f} minutes"
)

print(
    "\nSaved: "
    "mordred_mrmr_xgboost_validation_results.csv"
)

print(
    "The test set has not been used."
)

In [ ]:
import time
import pandas as pd

MRMR_MAX_FEATURES_EXTENDED = 300

start_time = time.time()

mrmr_ranked_features_300 = mrmr_classif(
    X=X_train_mordred_filtered,
    y=y_train_mordred,
    K=MRMR_MAX_FEATURES_EXTENDED,
    relevance="f",
    redundancy="c",
    n_jobs=4,
    show_progress=True
)

elapsed_minutes = (
    time.time() - start_time
) / 60


mrmr_ranking_300_df = pd.DataFrame({
    "rank": range(
        1,
        len(mrmr_ranked_features_300) + 1
    ),
    "descriptor": mrmr_ranked_features_300
})


mrmr_ranking_300_df.to_csv(
    "mordred_mrmr_top300_ranking.csv",
    index=False
)


print(
    "Number of ranked descriptors:",
    len(mrmr_ranked_features_300)
)

print(
    f"Time taken: {elapsed_minutes:.2f} minutes"
)

print("\nDescriptors ranked 101–120:")
display(
    mrmr_ranking_300_df.iloc[100:120]
)

print(
    "\nSaved: mordred_mrmr_top300_ranking.csv"
)

print(
    "Only the training set was used for mRMR."
)

In [ ]:
import time
import joblib
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    confusion_matrix,
    f1_score
)


extended_feature_counts = [
    150,
    200,
    250,
    300
]

extended_mrmr_results = []

start_time = time.time()


for feature_count in extended_feature_counts:

    print(
        f"Training XGBoost using top "
        f"{feature_count} mRMR descriptors..."
    )

    selected_features = (
        mrmr_ranked_features_300[
            :feature_count
        ]
    )

    X_train_selected = (
        X_train_mordred_filtered[
            selected_features
        ]
    )

    X_validation_selected = (
        X_validation_mordred_filtered[
            selected_features
        ]
    )


    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=3,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.10,
        reg_lambda=1.00,
        tree_method="hist",
        eval_metric="logloss",
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_train_selected,
        y_train_mordred,
        sample_weight=mordred_sample_weights,
        eval_set=[
            (
                X_validation_selected,
                y_validation_mordred
            )
        ],
        verbose=False
    )


    validation_probability = (
        model.predict_proba(
            X_validation_selected
        )[:, 1]
    )

    threshold = 0.50

    validation_prediction = (
        validation_probability >= threshold
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_validation_mordred,
        validation_prediction
    ).ravel()

    specificity = tn / (tn + fp)


    extended_mrmr_results.append({
        "model":
            f"Mordred mRMR XGBoost top {feature_count}",

        "number_of_features":
            feature_count,

        "threshold":
            threshold,

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation_mordred,
                validation_prediction
            ),

        "mcc":
            matthews_corrcoef(
                y_validation_mordred,
                validation_prediction
            ),

        "roc_auc":
            roc_auc_score(
                y_validation_mordred,
                validation_probability
            ),

        "average_precision":
            average_precision_score(
                y_validation_mordred,
                validation_probability
            ),

        "precision":
            precision_score(
                y_validation_mordred,
                validation_prediction,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_validation_mordred,
                validation_prediction,
                zero_division=0
            ),

        "specificity":
            specificity,

        "f1_score":
            f1_score(
                y_validation_mordred,
                validation_prediction,
                zero_division=0
            ),

        "true_negative":
            tn,

        "false_positive":
            fp,

        "false_negative":
            fn,

        "true_positive":
            tp,

        "best_iteration":
            model.best_iteration
    })


    joblib.dump(
        model,
        (
            f"mordred_mrmr_xgboost_"
            f"top{feature_count}_model.joblib"
        )
    )


    pd.DataFrame({
        "rank": range(
            1,
            feature_count + 1
        ),
        "descriptor": selected_features
    }).to_csv(
        (
            f"mordred_mrmr_top"
            f"{feature_count}_descriptors.csv"
        ),
        index=False
    )


    print(
        f"Completed top {feature_count} features."
    )


extended_mrmr_results_df = pd.DataFrame(
    extended_mrmr_results
)


# Combine with the previous 20–100 feature results
all_mrmr_feature_results_df = pd.concat(
    [
        mrmr_validation_results_df,
        extended_mrmr_results_df
    ],
    ignore_index=True
)


all_mrmr_feature_results_df = (
    all_mrmr_feature_results_df
    .drop_duplicates(
        subset=["number_of_features"],
        keep="last"
    )
    .sort_values(
        by=[
            "balanced_accuracy",
            "mcc"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


all_mrmr_feature_results_df.to_csv(
    "mordred_mrmr_all_feature_counts_results.csv",
    index=False
)


display(
    all_mrmr_feature_results_df.round(3)
)


elapsed_minutes = (
    time.time() - start_time
) / 60


print(
    f"\nTotal time: "
    f"{elapsed_minutes:.2f} minutes"
)

print(
    "\nSaved: "
    "mordred_mrmr_all_feature_counts_results.csv"
)

print(
    "The test set has not been used."
)

In [ ]:
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)


# Load the saved top-300 mRMR model
mordred_mrmr_top300_model = joblib.load(
    "mordred_mrmr_xgboost_top300_model.joblib"
)

top300_features = mrmr_ranked_features_300[:300]


# Generate validation probabilities
baseline_probability = (
    mordred_baseline_model.predict_proba(
        X_validation_mordred_filtered
    )[:, 1]
)

top300_probability = (
    mordred_mrmr_top300_model.predict_proba(
        X_validation_mordred_filtered[
            top300_features
        ]
    )[:, 1]
)


models_to_tune = {
    "Mordred XGBoost baseline - 626 features":
        baseline_probability,

    "Mordred mRMR XGBoost - 300 features":
        top300_probability
}


threshold_results = []


for model_name, probabilities in models_to_tune.items():

    for threshold in np.arange(
        0.20,
        0.81,
        0.01
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_validation_mordred,
            predictions
        ).ravel()

        specificity = tn / (tn + fp)

        threshold_results.append({
            "model": model_name,
            "threshold": round(
                float(threshold),
                2
            ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    y_validation_mordred,
                    predictions
                ),

            "mcc":
                matthews_corrcoef(
                    y_validation_mordred,
                    predictions
                ),

            "roc_auc":
                roc_auc_score(
                    y_validation_mordred,
                    probabilities
                ),

            "average_precision":
                average_precision_score(
                    y_validation_mordred,
                    probabilities
                ),

            "precision":
                precision_score(
                    y_validation_mordred,
                    predictions,
                    zero_division=0
                ),

            "recall":
                recall_score(
                    y_validation_mordred,
                    predictions,
                    zero_division=0
                ),

            "specificity":
                specificity,

            "f1_score":
                f1_score(
                    y_validation_mordred,
                    predictions,
                    zero_division=0
                ),

            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp
        })


mordred_threshold_results_df = pd.DataFrame(
    threshold_results
)


best_threshold_rows = []


for model_name in models_to_tune:

    model_results = (
        mordred_threshold_results_df[
            mordred_threshold_results_df[
                "model"
            ] == model_name
        ]
    )

    best_balanced_accuracy = (
        model_results
        .sort_values(
            by="balanced_accuracy",
            ascending=False
        )
        .iloc[0]
        .to_dict()
    )

    best_balanced_accuracy[
        "selection_metric"
    ] = "Balanced accuracy"

    best_mcc = (
        model_results
        .sort_values(
            by="mcc",
            ascending=False
        )
        .iloc[0]
        .to_dict()
    )

    best_mcc[
        "selection_metric"
    ] = "MCC"

    best_threshold_rows.extend([
        best_balanced_accuracy,
        best_mcc
    ])


best_mordred_thresholds_df = pd.DataFrame(
    best_threshold_rows
)


best_mordred_thresholds_df = (
    best_mordred_thresholds_df[
        [
            "model",
            "selection_metric",
            "threshold",
            "balanced_accuracy",
            "mcc",
            "roc_auc",
            "average_precision",
            "precision",
            "recall",
            "specificity",
            "f1_score",
            "true_negative",
            "false_positive",
            "false_negative",
            "true_positive"
        ]
    ]
)


display(
    best_mordred_thresholds_df.round(3)
)


mordred_threshold_results_df.to_csv(
    "mordred_threshold_tuning_all_results.csv",
    index=False
)

best_mordred_thresholds_df.to_csv(
    "mordred_best_threshold_comparison.csv",
    index=False
)


print(
    "Saved: mordred_threshold_tuning_all_results.csv"
)

print(
    "Saved: mordred_best_threshold_comparison.csv"
)

print(
    "\nThe test set has not been used."
)

In [ ]:
import pandas as pd


# Load the previous RDKit classification comparison
rdkit_results_df = pd.read_csv(
    "final_classification_validation_comparison.csv"
)


# Keep the two relevant RDKit XGBoost results
rdkit_xgboost_comparison = (
    rdkit_results_df[
        rdkit_results_df["model"].isin([
            "XGBoost - tuned threshold",
            "XGBoost - default threshold"
        ])
    ]
    .copy()
)

rdkit_xgboost_comparison[
    "descriptor_method"
] = "RDKit"

rdkit_xgboost_comparison[
    "number_of_features"
] = 177

rdkit_xgboost_comparison[
    "selection_method"
] = "No mRMR"


# Prepare the best Mordred threshold results
mordred_comparison = (
    best_mordred_thresholds_df.copy()
)

mordred_comparison[
    "descriptor_method"
] = "Mordred"

mordred_comparison[
    "number_of_features"
] = mordred_comparison[
    "model"
].apply(
    lambda name: 300
    if "300 features" in name
    else 626
)

mordred_comparison[
    "selection_method"
] = mordred_comparison[
    "model"
].apply(
    lambda name: "mRMR"
    if "mRMR" in name
    else "No mRMR"
)


# Make model names clearer
mordred_comparison["model"] = (
    mordred_comparison["model"]
    + " - "
    + mordred_comparison["selection_metric"]
)


comparison_columns = [
    "descriptor_method",
    "selection_method",
    "model",
    "number_of_features",
    "threshold",
    "balanced_accuracy",
    "mcc",
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "specificity",
    "f1_score",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive"
]


final_descriptor_comparison_df = pd.concat(
    [
        rdkit_xgboost_comparison[
            comparison_columns
        ],
        mordred_comparison[
            comparison_columns
        ]
    ],
    ignore_index=True
)


final_descriptor_comparison_df = (
    final_descriptor_comparison_df
    .sort_values(
        by=[
            "balanced_accuracy",
            "mcc"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    final_descriptor_comparison_df.round(3)
)


final_descriptor_comparison_df.to_csv(
    "final_rdkit_mordred_mrmr_validation_comparison.csv",
    index=False
)


print("\nBest balanced-accuracy result:")

display(
    final_descriptor_comparison_df
    .sort_values(
        "balanced_accuracy",
        ascending=False
    )
    .head(1)
    .round(3)
)


print("\nBest MCC result:")

display(
    final_descriptor_comparison_df
    .sort_values(
        "mcc",
        ascending=False
    )
    .head(1)
    .round(3)
)


print(
    "\nSaved: "
    "final_rdkit_mordred_mrmr_validation_comparison.csv"
)

print(
    "The test set has not been used."
)

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
import pandas as pd

# Freeze the final choices based only on validation results

FINAL_FEATURE_COUNT = 300
FINAL_THRESHOLD = 0.70
FINAL_N_ESTIMATORS = 1000

final_selected_features = list(
    mrmr_ranked_features_300[
        :FINAL_FEATURE_COUNT
    ]
)


# Combine training and validation features
# Only the already-selected 300 descriptors are used

X_train_validation_final = pd.concat(
    [
        X_train_mordred_filtered[
            final_selected_features
        ],

        X_validation_mordred_filtered[
            final_selected_features
        ]
    ],
    axis=0
).sort_index().astype("float32")


# Combine training and validation labels

y_train_validation_final = pd.concat(
    [
        y_train_mordred,
        y_validation_mordred
    ],
    axis=0
).sort_index().astype(int)


# Recalculate balanced weights for the combined data

final_training_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_validation_final
)


# Save the final descriptor list

pd.DataFrame({
    "rank": range(
        1,
        FINAL_FEATURE_COUNT + 1
    ),
    "descriptor": final_selected_features
}).to_csv(
    "final_mordred_mrmr_300_descriptors.csv",
    index=False
)


# Save the final model-selection decision

final_model_selection_df = pd.DataFrame([{
    "descriptor_method": "Mordred",
    "feature_selection": "mRMR",
    "model": "XGBoost",
    "number_of_features": FINAL_FEATURE_COUNT,
    "threshold": FINAL_THRESHOLD,
    "n_estimators": FINAL_N_ESTIMATORS,
    "validation_balanced_accuracy": 0.880,
    "validation_mcc": 0.680,
    "validation_roc_auc": 0.951,
    "validation_average_precision": 0.986,
    "validation_precision": 0.968,
    "validation_recall": 0.874,
    "validation_specificity": 0.886,
    "validation_f1_score": 0.919,
    "selection_reason":
        "Nearly identical performance to the 626-feature "
        "Mordred baseline while using 326 fewer descriptors."
}])


final_model_selection_df.to_csv(
    "final_model_selection_before_test.csv",
    index=False
)


print(
    "Combined feature shape:",
    X_train_validation_final.shape
)

print(
    "Combined label shape:",
    y_train_validation_final.shape
)

print(
    "Number of final descriptors:",
    len(final_selected_features)
)

print(
    "Remaining missing values:",
    int(
        X_train_validation_final
        .isna()
        .sum()
        .sum()
    )
)

print("\nCombined class counts:")
print(
    y_train_validation_final
    .value_counts()
    .sort_index()
)

print(
    "\nSaved: final_mordred_mrmr_300_descriptors.csv"
)

print(
    "Saved: final_model_selection_before_test.csv"
)

print(
    "\nThe final model choice is now fixed."
)

print(
    "The test set has not been used."
)

In [ ]:
import gc
import joblib
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    confusion_matrix,
    f1_score
)


# 1. Build the untouched test feature matrix

test_parts = []

for chunk_number, file_path in enumerate(
    numeric_chunk_files,
    start=1
):

    chunk_df = pd.read_pickle(file_path)

    test_mask = (
        data_df.loc[
            chunk_df.index,
            "split"
        ]
        .eq("test")
        .to_numpy()
    )

    test_part = chunk_df.loc[
        test_mask,
        final_selected_features
    ].copy()

    test_parts.append(test_part)

    del chunk_df
    gc.collect()

    if (
        chunk_number % 10 == 0
        or chunk_number == len(numeric_chunk_files)
    ):
        print(
            f"Loaded {chunk_number}/"
            f"{len(numeric_chunk_files)} chunks"
        )


X_test_final = (
    pd.concat(test_parts)
    .sort_index()
)

del test_parts
gc.collect()


# Fill test missing values using TRAINING medians only

X_test_final = (
    X_test_final
    .fillna(
        mordred_training_medians[
            final_selected_features
        ]
    )
    .astype("float32")
)


y_test_final = (
    data_df.loc[
        X_test_final.index,
        "dual_candidate"
    ]
    .astype(int)
)


print(
    "\nTest feature shape:",
    X_test_final.shape
)

print(
    "Test label shape:",
    y_test_final.shape
)

print(
    "Remaining missing values:",
    int(
        X_test_final
        .isna()
        .sum()
        .sum()
    )
)


# 2. Train final XGBoost on train + validation

final_mordred_mrmr_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=FINAL_N_ESTIMATORS,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


final_mordred_mrmr_model.fit(
    X_train_validation_final,
    y_train_validation_final,
    sample_weight=final_training_weights,
    verbose=False
)


# 3. Final untouched test predictions

final_test_probability = (
    final_mordred_mrmr_model.predict_proba(
        X_test_final
    )[:, 1]
)


final_test_prediction = (
    final_test_probability
    >= FINAL_THRESHOLD
).astype(int)


tn, fp, fn, tp = confusion_matrix(
    y_test_final,
    final_test_prediction
).ravel()

specificity = tn / (tn + fp)


# 4. Final test results

final_test_results_df = pd.DataFrame([{
    "descriptor_method": "Mordred",
    "feature_selection": "mRMR",
    "model": "XGBoost",
    "number_of_features": FINAL_FEATURE_COUNT,
    "threshold": FINAL_THRESHOLD,
    "training_rows":
        len(X_train_validation_final),
    "test_rows":
        len(X_test_final),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test_final,
            final_test_prediction
        ),

    "mcc":
        matthews_corrcoef(
            y_test_final,
            final_test_prediction
        ),

    "roc_auc":
        roc_auc_score(
            y_test_final,
            final_test_probability
        ),

    "average_precision":
        average_precision_score(
            y_test_final,
            final_test_probability
        ),

    "precision":
        precision_score(
            y_test_final,
            final_test_prediction,
            zero_division=0
        ),

    "recall":
        recall_score(
            y_test_final,
            final_test_prediction,
            zero_division=0
        ),

    "specificity":
        specificity,

    "f1_score":
        f1_score(
            y_test_final,
            final_test_prediction,
            zero_division=0
        ),

    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp
}])


display(
    final_test_results_df.round(3)
)


# --------------------------------------------------
# 5. Save model, predictions and results
# --------------------------------------------------

final_test_predictions_df = pd.DataFrame({
    "row_index":
        X_test_final.index,

    "canonical_smiles":
        data_df.loc[
            X_test_final.index,
            "canonical_smiles"
        ].values,

    "true_label":
        y_test_final.values,

    "predicted_probability":
        final_test_probability,

    "predicted_label":
        final_test_prediction,

    "threshold":
        FINAL_THRESHOLD
})


joblib.dump(
    final_mordred_mrmr_model,
    "final_mordred_mrmr_xgboost_model.joblib"
)

final_test_results_df.to_csv(
    "final_mordred_mrmr_test_results.csv",
    index=False
)

final_test_predictions_df.to_csv(
    "final_mordred_mrmr_test_predictions.csv",
    index=False
)


print(
    "\nSaved: final_mordred_mrmr_xgboost_model.joblib"
)

print(
    "Saved: final_mordred_mrmr_test_results.csv"
)

print(
    "Saved: final_mordred_mrmr_test_predictions.csv"
)

print(
    "\nThe untouched test set has now been evaluated once."
)

print(
    "Do not tune the model or threshold using these test results."
)

In [ ]:
import pandas as pd


# Load the frozen validation selection and final test result

validation_selection_df = pd.read_csv(
    "final_model_selection_before_test.csv"
)

test_result_df = pd.read_csv(
    "final_mordred_mrmr_test_results.csv"
)


validation_row = {
    "dataset": "Validation",
    "descriptor_method": "Mordred",
    "feature_selection": "mRMR",
    "model": "XGBoost",
    "number_of_features": 300,
    "threshold": 0.70,

    "balanced_accuracy":
        validation_selection_df.loc[
            0,
            "validation_balanced_accuracy"
        ],

    "mcc":
        validation_selection_df.loc[
            0,
            "validation_mcc"
        ],

    "roc_auc":
        validation_selection_df.loc[
            0,
            "validation_roc_auc"
        ],

    "average_precision":
        validation_selection_df.loc[
            0,
            "validation_average_precision"
        ],

    "precision":
        validation_selection_df.loc[
            0,
            "validation_precision"
        ],

    "recall":
        validation_selection_df.loc[
            0,
            "validation_recall"
        ],

    "specificity":
        validation_selection_df.loc[
            0,
            "validation_specificity"
        ],

    "f1_score":
        validation_selection_df.loc[
            0,
            "validation_f1_score"
        ]
}


test_row = {
    "dataset": "Untouched scaffold test",
    "descriptor_method":
        test_result_df.loc[
            0,
            "descriptor_method"
        ],

    "feature_selection":
        test_result_df.loc[
            0,
            "feature_selection"
        ],

    "model":
        test_result_df.loc[
            0,
            "model"
        ],

    "number_of_features":
        test_result_df.loc[
            0,
            "number_of_features"
        ],

    "threshold":
        test_result_df.loc[
            0,
            "threshold"
        ],

    "balanced_accuracy":
        test_result_df.loc[
            0,
            "balanced_accuracy"
        ],

    "mcc":
        test_result_df.loc[
            0,
            "mcc"
        ],

    "roc_auc":
        test_result_df.loc[
            0,
            "roc_auc"
        ],

    "average_precision":
        test_result_df.loc[
            0,
            "average_precision"
        ],

    "precision":
        test_result_df.loc[
            0,
            "precision"
        ],

    "recall":
        test_result_df.loc[
            0,
            "recall"
        ],

    "specificity":
        test_result_df.loc[
            0,
            "specificity"
        ],

    "f1_score":
        test_result_df.loc[
            0,
            "f1_score"
        ]
}


final_validation_test_comparison_df = pd.DataFrame([
    validation_row,
    test_row
])


display(
    final_validation_test_comparison_df.round(3)
)


final_validation_test_comparison_df.to_csv(
    "final_validation_vs_test_comparison.csv",
    index=False
)


print(
    "Saved: final_validation_vs_test_comparison.csv"
)

print(
    "\nThe final test evaluation is complete and frozen."
)

In [ ]:
import pandas as pd


# Extract importance scores from the final trained XGBoost model

booster_importance = (
    final_mordred_mrmr_model
    .get_booster()
    .get_score(
        importance_type="gain"
    )
)


final_feature_importance_df = pd.DataFrame({
    "descriptor": final_selected_features
})


final_feature_importance_df[
    "gain_importance"
] = (
    final_feature_importance_df[
        "descriptor"
    ]
    .map(booster_importance)
    .fillna(0)
)


final_feature_importance_df = (
    final_feature_importance_df
    .sort_values(
        by="gain_importance",
        ascending=False
    )
    .reset_index(drop=True)
)


final_feature_importance_df[
    "importance_rank"
] = range(
    1,
    len(final_feature_importance_df) + 1
)


final_feature_importance_df = (
    final_feature_importance_df[
        [
            "importance_rank",
            "descriptor",
            "gain_importance"
        ]
    ]
)


display(
    final_feature_importance_df.head(30)
)


final_feature_importance_df.to_csv(
    "final_mordred_descriptor_importance.csv",
    index=False
)


final_feature_importance_df.head(30).to_csv(
    "final_top30_mordred_descriptor_importance.csv",
    index=False
)


print(
    "Descriptors used by at least one tree:",
    (
        final_feature_importance_df[
            "gain_importance"
        ] > 0
    ).sum()
)

print(
    "\nSaved: final_mordred_descriptor_importance.csv"
)

print(
    "Saved: final_top30_mordred_descriptor_importance.csv"
)

In [ ]:
# Correctly extract and map XGBoost feature importance

import pandas as pd

final_booster = final_mordred_mrmr_model.get_booster()

raw_gain_importance = final_booster.get_score(
    importance_type="gain"
)

print(
    "Number of features returned by XGBoost:",
    len(raw_gain_importance)
)

print(
    "First 10 raw importance entries:"
)
print(
    list(raw_gain_importance.items())[:10]
)


mapped_gain_importance = {}

for feature_key, importance_value in raw_gain_importance.items():

    # Case 1: XGBoost retained the actual descriptor name
    if feature_key in final_selected_features:

        mapped_gain_importance[
            feature_key
        ] = importance_value

    # Case 2: XGBoost stored features as f0, f1, f2, etc.
    elif (
        feature_key.startswith("f")
        and feature_key[1:].isdigit()
    ):

        feature_index = int(
            feature_key[1:]
        )

        if feature_index < len(
            final_selected_features
        ):

            descriptor_name = (
                final_selected_features[
                    feature_index
                ]
            )

            mapped_gain_importance[
                descriptor_name
            ] = importance_value


final_feature_importance_df = pd.DataFrame({
    "descriptor": final_selected_features
})


final_feature_importance_df[
    "gain_importance"
] = (
    final_feature_importance_df[
        "descriptor"
    ]
    .map(mapped_gain_importance)
    .fillna(0.0)
)


# Normalised percentage importance

total_gain = final_feature_importance_df[
    "gain_importance"
].sum()

if total_gain > 0:

    final_feature_importance_df[
        "gain_importance_percent"
    ] = (
        final_feature_importance_df[
            "gain_importance"
        ]
        / total_gain
        * 100
    )

else:

    final_feature_importance_df[
        "gain_importance_percent"
    ] = 0.0


final_feature_importance_df = (
    final_feature_importance_df
    .sort_values(
        by="gain_importance",
        ascending=False
    )
    .reset_index(drop=True)
)


final_feature_importance_df[
    "importance_rank"
] = range(
    1,
    len(final_feature_importance_df) + 1
)


final_feature_importance_df = (
    final_feature_importance_df[
        [
            "importance_rank",
            "descriptor",
            "gain_importance",
            "gain_importance_percent"
        ]
    ]
)


display(
    final_feature_importance_df
    .head(30)
    .round(4)
)


features_used_count = (
    final_feature_importance_df[
        "gain_importance"
    ] > 0
).sum()


print(
    "\nDescriptors used by at least one tree:",
    features_used_count
)


# Overwrite the incorrect files

final_feature_importance_df.to_csv(
    "final_mordred_descriptor_importance.csv",
    index=False
)

final_feature_importance_df.head(30).to_csv(
    "final_top30_mordred_descriptor_importance.csv",
    index=False
)


print(
    "\nCorrected importance files saved."
)

In [ ]:
import matplotlib.pyplot as plt

top20_importance_df = (
    final_feature_importance_df
    .head(20)
    .sort_values(
        by="gain_importance_percent",
        ascending=True
    )
)

plt.figure(figsize=(10, 8))

plt.barh(
    top20_importance_df["descriptor"],
    top20_importance_df[
        "gain_importance_percent"
    ]
)

plt.xlabel("Gain importance (%)")
plt.ylabel("Mordred descriptor")
plt.title(
    "Top 20 Mordred Descriptors in the Final mRMR-XGBoost Model"
)

plt.tight_layout()

plt.savefig(
    "final_top20_mordred_descriptor_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


unused_descriptors_df = (
    final_feature_importance_df[
        final_feature_importance_df[
            "gain_importance"
        ] == 0
    ]
    .copy()
)

unused_descriptors_df.to_csv(
    "final_unused_mordred_descriptors.csv",
    index=False
)

print(
    "Number of unused descriptors:",
    len(unused_descriptors_df)
)

print("\nUnused descriptors:")
display(unused_descriptors_df)

print(
    "\nSaved: "
    "final_top20_mordred_descriptor_importance.png"
)

print(
    "Saved: final_unused_mordred_descriptors.csv"
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(
        y_test_final,
        final_test_prediction
    ),
    display_labels=[
        "Non-dual candidate",
        "Dual candidate"
    ]
)

fig, ax = plt.subplots(figsize=(7, 6))

confusion_display.plot(
    ax=ax,
    values_format="d"
)

ax.set_title(
    "Final mRMR-XGBoost Confusion Matrix\n"
    "Untouched Scaffold Test Set"
)

plt.tight_layout()

plt.savefig(
    "final_test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved: final_test_confusion_matrix.png")

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    auc
)


# ROC curve values
false_positive_rate, true_positive_rate, _ = roc_curve(
    y_test_final,
    final_test_probability
)

test_roc_auc = auc(
    false_positive_rate,
    true_positive_rate
)


plt.figure(figsize=(7, 6))

plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"Final model (AUC = {test_roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(
    "ROC Curve – Final mRMR-XGBoost Model\n"
    "Untouched Scaffold Test Set"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    "final_test_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# Precision–recall curve values
precision_values, recall_values, _ = (
    precision_recall_curve(
        y_test_final,
        final_test_probability
    )
)

test_pr_auc = auc(
    recall_values,
    precision_values
)


plt.figure(figsize=(7, 6))

plt.plot(
    recall_values,
    precision_values,
    label=f"Final model (AUC = {test_pr_auc:.3f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(
    "Precision–Recall Curve – Final mRMR-XGBoost Model\n"
    "Untouched Scaffold Test Set"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    "final_test_precision_recall_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print(
    f"Test ROC-AUC: {test_roc_auc:.3f}"
)

print(
    f"Test precision–recall AUC: {test_pr_auc:.3f}"
)

print(
    "\nSaved: final_test_roc_curve.png"
)

print(
    "Saved: final_test_precision_recall_curve.png"
)